# Importação de bibliotecas necessárias

Esta parte prepara todo o ambiente de desenvolvimento necessário para:

- Carregar e manipular dados
- Fazer pré-processamento avançado
- Treinar modelos de ML
- Avaliar performance
- Realizar validação cruzada

É essencialmente a "configuração inicial" que disponibiliza todas as ferramentas que serão usadas nas células seguintes do notebook.

In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder, LabelEncoder, QuantileTransformer
from sklearn.impute import SimpleImputer, KNNImputer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report
from sklearn.model_selection import RandomizedSearchCV, StratifiedKFold
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier, GradientBoostingClassifier, VotingClassifier
from sklearn.linear_model import LogisticRegression
import numpy as np
import warnings
warnings.filterwarnings('ignore')

# Carregamento e Análise Detalhada dos Dados

Essa parte realiza o carregamento inicial dos dados e uma análise exploratória abrangente para compreender a estrutura, qualidade e características do dataset antes de iniciar o processo de modelagem. É essencial para estabelecer uma base sólida de conhecimento sobre os dados, permitindo decisões informadas e estratégias otimizadas nas etapas subsequentes do pipeline de machine learning.

In [2]:
# Lê os arquivos CSV de treino, teste e submissão de exemplo
train_df = pd.read_csv("../database/train.csv")
test_df = pd.read_csv("../database/test.csv")
sample_submission_df = pd.read_csv("../database/sample_submission.csv")

# Exibe o formato (shape) dos conjuntos de dados
print("=== ANÁLISE INICIAL DOS DADOS ===")
print(f"Shape treino: {train_df.shape}")
print(f"Shape teste: {test_df.shape}")

# Analisa valores nulos em cada coluna dos datasets de treino e teste
print("\n=== ANÁLISE DE VALORES NULOS ===")
train_nulls = train_df.isnull().sum()
test_nulls = test_df.isnull().sum()

print("TREINO - Valores nulos por coluna:")
for col in train_nulls[train_nulls > 0].index:
    pct = (train_nulls[col] / len(train_df)) * 100
    print(f"  {col}: {train_nulls[col]} ({pct:.2f}%)")

print("\nTESTE - Valores nulos por coluna:")
for col in test_nulls[test_nulls > 0].index:
    pct = (test_nulls[col] / len(test_df)) * 100
    print(f"  {col}: {test_nulls[col]} ({pct:.2f}%)")

# Analisa a distribuição da variável alvo (labels) no conjunto de treino
print(f"\n=== DISTRIBUIÇÃO DA VARIÁVEL ALVO ===")
target_dist = train_df['labels'].value_counts(normalize=True)
print(f"Classe 0: {target_dist[0]:.3f} ({train_df['labels'].value_counts()[0]} amostras)")
print(f"Classe 1: {target_dist[1]:.3f} ({train_df['labels'].value_counts()[1]} amostras)")

# Exibe estatísticas descritivas das variáveis numéricas do conjunto de treino
print(f"\n=== ESTATÍSTICAS DESCRITIVAS ===")
numeric_cols = train_df.select_dtypes(include=[np.number]).columns
print("Variáveis numéricas:")
for col in numeric_cols:
    if col != 'labels':
        print(f"  {col}: min={train_df[col].min():.2f}, max={train_df[col].max():.2f}, "
              f"mean={train_df[col].mean():.2f}, std={train_df[col].std():.2f}")

print("Análise inicial concluída.")

=== ANÁLISE INICIAL DOS DADOS ===
Shape treino: (646, 33)
Shape teste: (277, 32)

=== ANÁLISE DE VALORES NULOS ===
TREINO - Valores nulos por coluna:
  age_first_funding_year: 35 (5.42%)
  age_last_funding_year: 9 (1.39%)
  age_first_milestone_year: 138 (21.36%)
  age_last_milestone_year: 111 (17.18%)

TESTE - Valores nulos por coluna:
  age_first_funding_year: 11 (3.97%)
  age_last_funding_year: 4 (1.44%)
  age_first_milestone_year: 60 (21.66%)
  age_last_milestone_year: 53 (19.13%)

=== DISTRIBUIÇÃO DA VARIÁVEL ALVO ===
Classe 0: 0.353 (228 amostras)
Classe 1: 0.647 (418 amostras)

=== ESTATÍSTICAS DESCRITIVAS ===
Variáveis numéricas:
  id: min=1.00, max=923.00, mean=461.58, std=264.86
  age_first_funding_year: min=0.00, max=21.90, mean=2.34, std=2.47
  age_last_funding_year: min=0.00, max=21.90, mean=4.04, std=2.95
  age_first_milestone_year: min=0.00, max=24.68, mean=3.35, std=2.87
  age_last_milestone_year: min=0.00, max=24.68, mean=4.94, std=3.21
  relationships: min=0.00, max=63

# Tratamento de Valores Nulos

Esta parte estabelece o padrão de qualidade dos dados que será utilizado em todas as etapas subsequentes, garantindo que o pipeline de machine learning opere com dados íntegros e semanticamente consistentes, maximizando o potencial preditivo do modelo final.

In [3]:
def advanced_null_treatment(df, is_train=True):
    """
    Tratamento inteligente de valores nulos baseado no tipo e distribuição dos dados.
    - Preenche valores nulos de acordo com o contexto de cada coluna.
    - Utiliza estratégias robustas para garantir integridade dos dados.
    """
    df = df.copy()
    
    print(f"Tratando valores nulos ({'TREINO' if is_train else 'TESTE'})...")
        
    # category_code: valores nulos recebem 'unknown'
    if 'category_code' in df.columns:
        df['category_code'] = df['category_code'].fillna('unknown')
        print("  - category_code: preenchido com 'unknown'")
    
    # Colunas de financiamento: nulo geralmente significa ausência de valor, então preenche com 0
    funding_cols = ['funding_total_usd', 'funding_rounds']
    for col in funding_cols:
        if col in df.columns and df[col].isnull().sum() > 0:
            df[col] = df[col].fillna(0)
            print(f"  - {col}: preenchido com 0")
    
    # Colunas de relacionamento e marcos: usa mediana para evitar distorção por outliers
    relationship_cols = ['relationships', 'milestones']
    for col in relationship_cols:
        if col in df.columns and df[col].isnull().sum() > 0:
            median_val = df[col].median()
            df[col] = df[col].fillna(median_val)
            print(f"  - {col}: preenchido com mediana ({median_val})")
    
    # avg_participants: utiliza KNN imputation para capturar padrões entre variáveis correlatas
    if 'avg_participants' in df.columns and df['avg_participants'].isnull().sum() > 0:
        aux_features = ['funding_total_usd', 'funding_rounds', 'milestones', 'relationships']
        aux_data = df[aux_features].fillna(0)
        knn_imputer = KNNImputer(n_neighbors=5)
        df['avg_participants'] = knn_imputer.fit_transform(df[['avg_participants']].join(aux_data))[:, 0]
        print("  - avg_participants: preenchido com KNN imputation")
    
    # Se ainda restarem nulos, preenche com valores padrão (mediana para numéricos, 'missing' para categóricos)
    remaining_nulls = df.isnull().sum().sum()
    if remaining_nulls > 0:
        print(f"  ATENÇÃO: {remaining_nulls} valores nulos restantes")
        for col in df.columns:
            if df[col].isnull().sum() > 0:
                if df[col].dtype in ['object']:
                    df[col] = df[col].fillna('missing')
                else:
                    df[col] = df[col].fillna(df[col].median())
    
    print(f"Tratamento de nulos concluído. Total de nulos restantes: {df.isnull().sum().sum()}")
    return df

# Aplica tratamento de valores nulos nos conjuntos de treino e teste
train_df = advanced_null_treatment(train_df, is_train=True)
test_df = advanced_null_treatment(test_df, is_train=False)


Tratando valores nulos (TREINO)...
  - category_code: preenchido com 'unknown'
  ATENÇÃO: 293 valores nulos restantes
Tratamento de nulos concluído. Total de nulos restantes: 0
Tratando valores nulos (TESTE)...
  - category_code: preenchido com 'unknown'
  ATENÇÃO: 128 valores nulos restantes
Tratamento de nulos concluído. Total de nulos restantes: 0


# Feature Engineering

Esta parte multiplica a capacidade preditiva dos dados originais através de engenharia de features, criando representações que capturam nuances do negócio de startups e investimentos, enquanto mantém robustez estatística e interpretabilidade para stakeholders.

In [4]:
def create_advanced_features(df_train, df_test=None):
    """
    Cria features avançadas para melhorar a capacidade preditiva do modelo.
    
    """
    train_df = df_train.copy()
    test_df = df_test.copy() if df_test is not None else None

    print(f"Criando features avançadas...")

    def process_single_df(df):
        # Calcula o financiamento por rodada (tratamento robusto para divisão por zero)
        df['funding_per_round'] = np.where(
            df['funding_rounds'] > 0,
            df['funding_total_usd'] / df['funding_rounds'],
            0
        )

        # Aplica transformações logarítmicas para reduzir skewness
        df['funding_total_usd_log'] = np.log1p(df['funding_total_usd'])
        df['avg_participants_log'] = np.log1p(df['avg_participants'])
        df['funding_per_round_log'] = np.log1p(df['funding_per_round'])

        # Cria interações geográficas com financiamento
        df['funding_ca_interaction'] = df['funding_total_usd'] * df['is_CA']
        df['funding_ny_interaction'] = df['funding_total_usd'] * df['is_NY']
        df['funding_other_states'] = df['funding_total_usd'] * (1 - df['is_CA'] - df['is_NY'])

        # Interações entre atividades
        df['rounds_relationships'] = df['funding_rounds'] * df['relationships']
        df['rounds_milestones'] = df['funding_rounds'] * df['milestones']
        df['relationships_milestones'] = df['relationships'] * df['milestones']

        # Interação participantes com atividades
        df['participants_rounds'] = df['avg_participants'] * df['funding_rounds']
        df['participants_funding'] = df['avg_participants'] * df['funding_total_usd_log']

        # Soma total de atividades da empresa
        df['total_activity'] = df['funding_rounds'] + df['milestones'] + df['relationships']
        df['total_activity_log'] = np.log1p(df['total_activity'])

        # Métricas de eficiência
        df['funding_efficiency'] = np.where(
            df['total_activity'] > 0,
            df['funding_total_usd'] / df['total_activity'],
            0
        )
        df['milestone_efficiency'] = np.where(
            df['funding_rounds'] > 0,
            df['milestones'] / df['funding_rounds'],
            0
        )
        df['relationship_efficiency'] = np.where(
            df['total_activity'] > 0,
            df['relationships'] / df['total_activity'],
            0
        )

        # Combina localização com setor
        df['location_sector'] = df['is_CA'].astype(str) + "_" + df['category_code'].astype(str)

        # Cria coluna de estado completo (CA, NY ou Others)
        df['state_full'] = np.where(df['is_CA'] == 1, 'CA',
                                   np.where(df['is_NY'] == 1, 'NY', 'Others'))

        # Combina localização com nível de atividade (quantis)
        activity_level = pd.qcut(df['total_activity'].rank(method='first'),
                                 q=3, labels=['low_activity', 'medium_activity', 'high_activity'])
        df['location_activity'] = df['state_full'] + "_" + activity_level.astype(str)

        # Rankings percentuais das principais métricas
        df['funding_rank'] = df['funding_total_usd'].rank(pct=True)
        df['participants_rank'] = df['avg_participants'].rank(pct=True)
        df['activity_rank'] = df['total_activity'].rank(pct=True)
        df['efficiency_rank'] = df['funding_efficiency'].rank(pct=True)

        # Scores compostos
        df['success_score'] = (df['funding_rank'] + df['activity_rank'] + df['participants_rank']) / 3
        df['efficiency_score'] = (df['funding_efficiency'] + df['milestone_efficiency']) / 2

        # Tiers de financiamento (quantis)
        df['funding_tier'] = pd.qcut(df['funding_total_usd'].rank(method='first'),
                                     q=5, labels=['very_low', 'low', 'medium', 'high', 'very_high'])

        # Tiers de participação (quantis)
        df['participation_tier'] = pd.qcut(df['avg_participants'].rank(method='first'),
                                           q=4, labels=['low_part', 'medium_part', 'high_part', 'very_high_part'])

        # Razões e proporções adicionais
        df['milestones_to_relationships_ratio'] = np.where(
            df['relationships'] > 0,
            df['milestones'] / df['relationships'],
            0
        )
        df['funding_to_participants_ratio'] = np.where(
            df['avg_participants'] > 0,
            df['funding_total_usd'] / df['avg_participants'],
            0
        )

        return df

    # Aplica processamento nos DataFrames de treino e teste
    train_df = process_single_df(train_df)
    if test_df is not None:
        test_df = process_single_df(test_df)

    # Calcula estatísticas por categoria no treino
    category_stats = train_df.groupby('category_code')['funding_total_usd'].agg(['mean', 'std', 'median']).reset_index()
    category_stats.columns = ['category_code', 'category_funding_mean', 'category_funding_std', 'category_funding_median']

    # Calcula estatísticas por estado no treino
    state_stats = train_df.groupby('state_full')['funding_total_usd'].agg(['mean', 'std']).reset_index()
    state_stats.columns = ['state_full', 'state_funding_mean', 'state_funding_std']

    # Junta estatísticas ao treino
    train_df = train_df.merge(category_stats, on='category_code', how='left')
    train_df = train_df.merge(state_stats, on='state_full', how='left')

    # Calcula desvios em relação à média da categoria e do estado
    train_df['funding_deviation_from_category'] = train_df['funding_total_usd'] - train_df['category_funding_mean']
    train_df['funding_deviation_from_state'] = train_df['funding_total_usd'] - train_df['state_funding_mean']

    # Aplica estatísticas do treino no teste
    if test_df is not None:
        test_df = test_df.merge(category_stats, on='category_code', how='left')
        test_df = test_df.merge(state_stats, on='state_full', how='left')

        # Preenche valores ausentes no teste com médias globais do treino
        test_df['category_funding_mean'].fillna(train_df['funding_total_usd'].mean(), inplace=True)
        test_df['category_funding_std'].fillna(train_df['funding_total_usd'].std(), inplace=True)
        test_df['category_funding_median'].fillna(train_df['funding_total_usd'].median(), inplace=True)
        test_df['state_funding_mean'].fillna(train_df['funding_total_usd'].mean(), inplace=True)
        test_df['state_funding_std'].fillna(train_df['funding_total_usd'].std(), inplace=True)

        # Calcula desvios no teste
        test_df['funding_deviation_from_category'] = test_df['funding_total_usd'] - test_df['category_funding_mean']
        test_df['funding_deviation_from_state'] = test_df['funding_total_usd'] - test_df['state_funding_mean']

    print(f"Features criadas! Train shape: {train_df.shape}")
    if test_df is not None:
        print(f"Test shape: {test_df.shape}")

    return train_df, test_df

# Aplica o feature engineering avançado
train_df_eng, test_df_eng = create_advanced_features(train_df, test_df)

def treat_outliers_advanced(df, is_train=True):
    """
    Tratamento inteligente de outliers usando múltiplas estratégias.
    Cap os valores extremos usando o método do IQR.
    """
    df = df.copy()

    if is_train:
        print("Tratando outliers no conjunto de treino...")

        # Lista de colunas numéricas para tratamento
        numeric_cols = ['funding_total_usd', 'avg_participants', 'milestones',
                       'funding_per_round', 'funding_efficiency']

        for col in numeric_cols:
            if col in df.columns:
                # Método IQR (Interquartile Range)
                Q1 = df[col].quantile(0.25)
                Q3 = df[col].quantile(0.75)
                IQR = Q3 - Q1

                # Define limites inferior e superior
                lower_bound = Q1 - 1.5 * IQR
                upper_bound = Q3 + 1.5 * IQR

                # Conta outliers
                outliers_count = ((df[col] < lower_bound) | (df[col] > upper_bound)).sum()

                if outliers_count > 0:
                    # Cap os outliers (substitui valores extremos pelos limites)
                    df[col] = np.where(df[col] > upper_bound, upper_bound, df[col])
                    df[col] = np.where(df[col] < lower_bound, lower_bound, df[col])
                    print(f"  - {col}: {outliers_count} outliers tratados")

    return df

# Aplica tratamento de outliers
train_df_eng = treat_outliers_advanced(train_df_eng, is_train=True)
test_df_eng = treat_outliers_advanced(test_df_eng, is_train=False)

# Remove features irrelevantes para o modelo
irrelevant_features = ['closed_at', 'status', 'is_top500', 'index', 'company_id', 'permalink']
for col in irrelevant_features:
    if col in train_df_eng.columns:
        train_df_eng = train_df_eng.drop(columns=[col])
    if col in test_df_eng.columns:
        test_df_eng = test_df_eng.drop(columns=[col])

print("\n=== RESUMO DO FEATURE ENGINEERING ===")
print(f"Features no conjunto de treino: {train_df_eng.shape[1]}")
print(f"Features no conjunto de teste: {test_df_eng.shape[1]}")
print("Feature Engineering avançado concluído!")

Criando features avançadas...
Features criadas! Train shape: (646, 70)
Test shape: (277, 69)
Tratando outliers no conjunto de treino...
  - funding_total_usd: 50 outliers tratados
  - avg_participants: 19 outliers tratados
  - funding_per_round: 46 outliers tratados
  - funding_efficiency: 60 outliers tratados

=== RESUMO DO FEATURE ENGINEERING ===
Features no conjunto de treino: 70
Features no conjunto de teste: 69
Feature Engineering avançado concluído!


# Pré-processamento 

Esta parte estabelece a fundação algorítmica que transforma features em formato otimizado para machine learning, aplicando transformações que maximizam performance, generalização e interpretabilidade dos modelos subsequentes.

In [5]:

# Salva os IDs do teste para submissão final
test_ids = test_df_eng["id"]

# Separa features e target
X = train_df_eng.drop(["labels", "id"], axis=1)
y = train_df_eng["labels"]
X_test = test_df_eng.drop("id", axis=1)

# Identifica tipos de features
numerical_features = X.select_dtypes(include=[np.number]).columns.tolist()
categorical_features = X.select_dtypes(include=['object', 'category']).columns.tolist()

print(f"=== PIPELINE DE PRÉ-PROCESSAMENTO ===")
print(f"Features numéricas: {len(numerical_features)}")
print(f"Features categóricas: {len(categorical_features)}")

# Pipeline robusto para features numéricas
numerical_pipeline = Pipeline([
    ('imputer', KNNImputer(n_neighbors=5)),  # KNN para capturar relações
    ('quantile_transformer', QuantileTransformer(output_distribution='normal')),  # Normalização robusta
    ('scaler', StandardScaler())  # Padronização final
])

# Pipeline para features categóricas
categorical_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False, max_categories=50))
])

# Combina pipelines
preprocessor = ColumnTransformer([
    ('num', numerical_pipeline, numerical_features),
    ('cat', categorical_pipeline, categorical_features)
], remainder='drop')

# Aplica pré-processamento
print("Aplicando pré-processamento robusto...")
X_processed = preprocessor.fit_transform(X)
X_test_processed = preprocessor.transform(X_test)

# Recupera nomes das features
try:
    cat_feature_names = preprocessor.named_transformers_['cat']['onehot'].get_feature_names_out(categorical_features)
    feature_names = numerical_features + list(cat_feature_names)
except:
    feature_names = [f'feature_{i}' for i in range(X_processed.shape[1])]

# Cria DataFrames finais
X_processed_df = pd.DataFrame(X_processed, columns=feature_names)
X_test_processed_df = pd.DataFrame(X_test_processed, columns=feature_names)

print(f"Shape final dos dados processados: {X_processed_df.shape}")
print(f"Total de features após pré-processamento: {X_processed_df.shape[1]}")

# Seleção de features importantes
print("\n=== SELEÇÃO DE FEATURES ===")
feature_selector = SelectKBest(score_func=f_classif, k=min(100, X_processed_df.shape[1]))
X_processed_selected = feature_selector.fit_transform(X_processed_df, y)
X_test_processed_selected = feature_selector.transform(X_test_processed_df)

selected_features = X_processed_df.columns[feature_selector.get_support()]
print(f"Features selecionadas: {len(selected_features)}")

# Cria DataFrames finais com features selecionadas
X_processed_df = pd.DataFrame(X_processed_selected, columns=selected_features)
X_test_processed_df = pd.DataFrame(X_test_processed_selected, columns=selected_features)

print("Pré-processamento avançado concluído!")

=== PIPELINE DE PRÉ-PROCESSAMENTO ===
Features numéricas: 62
Features categóricas: 6
Aplicando pré-processamento robusto...
Shape final dos dados processados: (646, 167)
Total de features após pré-processamento: 167

=== SELEÇÃO DE FEATURES ===
Features selecionadas: 100
Pré-processamento avançado concluído!


# Split para validação

Esta parte executa a divisão estratégica final dos dados processados, criando conjuntos de treino e validação balanceados que servirão como base para toda a etapa de modelagem.

In [6]:
# Divide os dados de treino em treino e validação para avaliar o modelo
X_train, X_val, y_train, y_val = train_test_split(
    X_processed_df, y, test_size=0.2, random_state=42, stratify=y
)

# Modelagem com RandomizedSearchCV + Múltiplos Algoritmos

Esta parte implementa o núcleo de inteligência artificial do projeto, onde múltiplos algoritmos competem automaticamente para encontrar a melhor solução preditiva, combinando otimização de hiperparâmetros, ensemble learning e seleção automática de modelos.

In [7]:
# ...existing code...

# Estratégia de validação
cv_strategy = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# Parâmetros para RandomForest
rf_param_dist = {
    "n_estimators": [300, 500, 700, 1000],
    "max_depth": [15, 20, 25, None],
    "min_samples_split": [2, 4, 6],
    "min_samples_leaf": [1, 2, 3],
    "max_features": ["sqrt", "log2", 0.5],
    "class_weight": ["balanced", None],
    "bootstrap": [True, False]
}

# Parâmetros para GradientBoosting
gb_param_dist = {
    "n_estimators": [300, 500, 800, 1000],
    "learning_rate": [0.01, 0.05, 0.1, 0.15],
    "max_depth": [4, 6, 8, 10],
    "min_samples_split": [2, 5, 10],
    "min_samples_leaf": [1, 2, 4],
    "subsample": [0.8, 0.9, 1.0],
    "max_features": ["sqrt", "log2", 0.8]
}

# Parâmetros para Regressão Logística
lr_param_dist = {
    "C": [0.001, 0.01, 0.1, 1, 10, 100],
    "penalty": ["l1", "l2", "elasticnet"],
    "solver": ["liblinear", "saga"],
    "class_weight": ["balanced", None],
    "max_iter": [1000, 2000, 3000]
}

print("=== TREINAMENTO DOS 3 MODELOS ===")

# 1. RandomForest
print("1. Otimizando RandomForest...")
rf_search = RandomizedSearchCV(
    estimator=RandomForestClassifier(random_state=42),
    param_distributions=rf_param_dist,
    n_iter=30,
    scoring="accuracy",
    cv=cv_strategy,
    random_state=42,
    n_jobs=-1,
    verbose=1
)
rf_search.fit(X_train, y_train)
rf_best = rf_search.best_estimator_
rf_val_pred = rf_best.predict(X_val)
rf_val_score = accuracy_score(y_val, rf_val_pred)
print(f"RandomForest - Acurácia: {rf_val_score:.4f}")

# 2. GradientBoosting
print("\n2. Otimizando GradientBoosting...")
gb_search = RandomizedSearchCV(
    estimator=GradientBoostingClassifier(random_state=42),
    param_distributions=gb_param_dist,
    n_iter=30,
    scoring="accuracy",
    cv=cv_strategy,
    random_state=42,
    n_jobs=-1,
    verbose=1
)
gb_search.fit(X_train, y_train)
gb_best = gb_search.best_estimator_
gb_val_pred = gb_best.predict(X_val)
gb_val_score = accuracy_score(y_val, gb_val_pred)
print(f"GradientBoosting - Acurácia: {gb_val_score:.4f}")

# 3. Regressão Logística
print("\n3. Otimizando Regressão Logística...")
lr_search = RandomizedSearchCV(
    estimator=LogisticRegression(random_state=42),
    param_distributions=lr_param_dist,
    n_iter=25,
    scoring="accuracy",
    cv=cv_strategy,
    random_state=42,
    n_jobs=-1,
    verbose=1
)
lr_search.fit(X_train, y_train)
lr_best = lr_search.best_estimator_
lr_val_pred = lr_best.predict(X_val)
lr_val_score = accuracy_score(y_val, lr_val_pred)
print(f"Regressão Logística - Acurácia: {lr_val_score:.4f}")

# =========================
# COMPARAÇÃO E SELEÇÃO
# =========================

models_results = {
    'RandomForest': (rf_best, rf_val_pred, rf_val_score),
    'GradientBoosting': (gb_best, gb_val_pred, gb_val_score),
    'LogisticRegression': (lr_best, lr_val_pred, lr_val_score)
}

print("\n=== COMPARAÇÃO DOS MODELOS ===")
for name, (model, pred, score) in models_results.items():
    print(f"{name}: {score:.4f}")

# Melhor modelo individual
best_model_name = max(models_results.items(), key=lambda x: x[1][2])[0]
best_individual_model = models_results[best_model_name][0]
best_individual_score = models_results[best_model_name][2]

print(f"\nMelhor Individual: {best_model_name} ({best_individual_score:.4f})")

# =========================
# ENSEMBLE
# =========================

# Ensemble com os 3 modelos
voting_ensemble = VotingClassifier(
    estimators=[
        ('rf', rf_best),
        ('gb', gb_best),
        ('lr', lr_best)
    ],
    voting='soft'
)

print("\nTreinando Ensemble...")
voting_ensemble.fit(X_train, y_train)
ensemble_val_pred = voting_ensemble.predict(X_val)
ensemble_val_score = accuracy_score(y_val, ensemble_val_pred)

print(f"✅ Ensemble - Acurácia: {ensemble_val_score:.4f}")

# =========================
# SELEÇÃO FINAL
# =========================

if ensemble_val_score > best_individual_score:
    final_model = voting_ensemble
    final_pred = ensemble_val_pred
    final_score = ensemble_val_score
    model_type = "Voting Ensemble (3 modelos)"
else:
    final_model = best_individual_model
    final_pred = models_results[best_model_name][1]
    final_score = best_individual_score
    model_type = f"Individual - {best_model_name}"

print(f"\n=== MODELO FINAL SELECIONADO ===")
print(f"Tipo: {model_type}")
print(f"Acurácia: {final_score:.4f}")

# Análise final
print(f"\n=== ANÁLISE FINAL ===")
from sklearn.metrics import confusion_matrix, classification_report

cm = confusion_matrix(y_val, final_pred)
print("Matriz de Confusão:")
print(cm)

print(f"\nRanking Final:")
all_scores = [
    ('Ensemble', ensemble_val_score),
    ('RandomForest', rf_val_score),
    ('GradientBoosting', gb_val_score),
    ('LogisticRegression', lr_val_score)
]

# Guarda variáveis para uso posterior
y_pred = final_pred
gb_model = final_model

=== TREINAMENTO DOS 3 MODELOS ===
1. Otimizando RandomForest...
Fitting 5 folds for each of 30 candidates, totalling 150 fits
RandomForest - Acurácia: 0.7769

2. Otimizando GradientBoosting...
Fitting 5 folds for each of 30 candidates, totalling 150 fits
GradientBoosting - Acurácia: 0.7846

3. Otimizando Regressão Logística...
Fitting 5 folds for each of 25 candidates, totalling 125 fits
Regressão Logística - Acurácia: 0.7615

=== COMPARAÇÃO DOS MODELOS ===
RandomForest: 0.7769
GradientBoosting: 0.7846
LogisticRegression: 0.7615

Melhor Individual: GradientBoosting (0.7846)

Treinando Ensemble...
✅ Ensemble - Acurácia: 0.7769

=== MODELO FINAL SELECIONADO ===
Tipo: Individual - GradientBoosting
Acurácia: 0.7846

=== ANÁLISE FINAL ===
Matriz de Confusão:
[[26 20]
 [ 8 76]]

Ranking Final:


# Previsão e submissão

Esta parte materializa toda a inteligência desenvolvida no pipeline em predições concretas para submissão, convertendo o modelo otimizado em resultados prontos para avaliação.

In [8]:
# Usa as variáveis já criadas no notebook (não carrega de arquivos CSV)
print("Gerando previsões para o conjunto de teste...")
predictions = gb_model.predict(X_test_processed_df)
print("Previsões geradas.")

# Cria o DataFrame de submissão no formato solicitado
submission_df = pd.DataFrame({
    "id": test_ids,
    "labels": predictions
})

# Salva o arquivo de submissão
submission_df.to_csv("submission.csv", index=False)
print("Arquivo de submissão criado: submission.csv")
print(f"Distribuição das previsões: {pd.Series(predictions).value_counts(normalize=True)}")

# Exibe algumas informações da submissão
print(f"Número total de previsões: {len(predictions)}")
print(f"Primeiras 10 previsões: {predictions[:10]}")
print(f"Shape do arquivo de submissão: {submission_df.shape}")

Gerando previsões para o conjunto de teste...
Previsões geradas.
Arquivo de submissão criado: submission.csv
Distribuição das previsões: 1    0.714801
0    0.285199
Name: proportion, dtype: float64
Número total de previsões: 277
Primeiras 10 previsões: [1 0 1 1 0 1 0 0 1 1]
Shape do arquivo de submissão: (277, 2)


# Validação Final de Performance

Esta parte serve como validação final consolidada que transforma toda a complexidade do sistema multi-algoritmo em uma métrica única e clara, fornecendo confirmação definitiva da qualidade e sucesso do pipeline de machine learning.

In [9]:
val_accuracy = accuracy_score(y_val, y_pred)
print(f"\nAcurácia no conjunto de validação: {val_accuracy:.4f}")


Acurácia no conjunto de validação: 0.7846
